# Data Science Capstone Project

## Project Title
Online Bookstore Data Analysis and Rating Prediction

## Project Objective
The objective of this project is to collect real-world data from a public website and transform it into structured, decision-ready insights. 

The workflow includes:
- Web scraping
- Data cleaning and preprocessing
- Exploratory data analysis
- Machine learning model development
- Interactive dashboard creation

The dataset used in this project is collected from the website **books.toscrape.com**, which provides publicly accessible information about books including price, rating, category, and availability.nfirmation message
print("\nWeb Scraping Completed Successfully")
print("Total records collected:", len(df))


# Display first few rows of dataset
df.head()

## Phase 1: Web Scraping

In this phase, book data is collected from the website **books.toscrape.com** using Python.

The following information is extracted for each book:

- Title
- Price
- Rating
- Availability status
- Category
- Stock count
- UPC (unique product identifier)
- Product link

The website contains 50 pages with approximately 20 books per page, resulting in around **1000 records**. 

Python libraries such as **Requests** and **BeautifulSoup** are used to send HTTP requests and parse HTML content. The collected data is stored in a structured format using **Pandas DataFrame** and saved as a CSV file for further analysis.

In [2]:
!pip install requests beautifulsoup4 pandas

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://books.toscrape.com/catalogue/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

data = []

for page in range(1, 51):

    print(f"Scraping page {page}/50")

    url = f"{BASE_URL}page-{page}.html"

    response = requests.get(url, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.h3.a["title"]

        price = book.find("p", class_="price_color").text
        price = price.replace("£","").replace("Â","").strip()
        price = float(price)

        rating = book.p["class"][1]

        availability = book.find("p", class_="instock availability").text.strip()

        relative_link = book.h3.a["href"]
        product_link = BASE_URL + relative_link

        try:
            book_page = requests.get(product_link, headers=HEADERS, timeout=10)
            book_soup = BeautifulSoup(book_page.text, "html.parser")

            category = book_soup.find("ul", class_="breadcrumb").find_all("li")[2].text.strip()

            table = book_soup.find("table", class_="table table-striped")
            rows = table.find_all("tr")

            upc = rows[0].td.text
            stock_text = rows[5].td.text

            stock_count = ''.join(filter(str.isdigit, stock_text))

        except:
            category = None
            upc = None
            stock_count = None

        data.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Category": category,
            "Stock_Count": stock_count,
            "UPC": upc,
            "Product_Link": product_link
        })

        if len(data) % 50 == 0:
            print(f"{len(data)} books collected")

        time.sleep(0.2)

df = pd.DataFrame(data)

df.to_csv("books_raw.csv", index=False)

print("\nScraping Completed")
print("Total books:", len(df))

df.head()

Scraping page 1/50
Scraping page 2/50
Scraping page 3/50
50 books collected
Scraping page 4/50
Scraping page 5/50
100 books collected
Scraping page 6/50
Scraping page 7/50
Scraping page 8/50
150 books collected
Scraping page 9/50
Scraping page 10/50
200 books collected
Scraping page 11/50
Scraping page 12/50
Scraping page 13/50
250 books collected
Scraping page 14/50
Scraping page 15/50
300 books collected
Scraping page 16/50
Scraping page 17/50
Scraping page 18/50
350 books collected
Scraping page 19/50
Scraping page 20/50
400 books collected
Scraping page 21/50
Scraping page 22/50
Scraping page 23/50
450 books collected
Scraping page 24/50
Scraping page 25/50
500 books collected
Scraping page 26/50
Scraping page 27/50
Scraping page 28/50
550 books collected
Scraping page 29/50
Scraping page 30/50
600 books collected
Scraping page 31/50
Scraping page 32/50
Scraping page 33/50
650 books collected
Scraping page 34/50
Scraping page 35/50
700 books collected
Scraping page 36/50
Scraping p

,Title,Price,Rating,Availability,Category,Stock_Count,UPC,Product_Link
0,A Light in the Attic,51.77,Three,In stock,Poetry,22,a897fe39b1053632,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,Historical Fiction,20,90fa61229261140a,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,Fiction,20,6957f44c3847a760,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,Mystery,20,e00eb4fd7b871a48,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,History,20,4165285e1663650f,https://books.toscrape.com/catalogue/sapiens-a...


In [6]:
df.describe()

,Price
count,1000.00000
mean,35.07035
std,14.44669
min,10.00000
25%,22.10750
50%,35.98000
75%,47.45750
max,59.99000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Title         1000 non-null   object 
 1   Price         1000 non-null   float64
 2   Rating        1000 non-null   object 
 3   Availability  1000 non-null   object 
 4   Category      1000 non-null   object 
 5   Stock_Count   1000 non-null   object 
 6   UPC           1000 non-null   object 
 7   Product_Link  1000 non-null   object 
dtypes: float64(1), object(7)
memory usage: 62.6+ KB


In [9]:
df.shape

(1000, 8)